**E1 - Procesamiento de datos con pandas**

Sesión 14 sep 2026. Dataset: Datos/ventas.csv

In [1]:
import json
from pathlib import Path

import pandas as pd

**Parte 1 - Diagnóstico**

In [2]:
DATA_DIR = Path("Datos")

ventas = pd.read_csv(DATA_DIR / "ventas.csv")
ventas

,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027
...,...,...,...,...,...,...
145,2026-02-15,Sur,Sensor A,16,19.50,C027
146,2026-01-13,Oeste,Hub IoT,19,85.00,C040
147,2026-02-09,Oeste,Sensor B,9,32.00,C009
148,2026-01-08,Oeste,Sensor B,5,32.31,C024


In [11]:
print(ventas.shape)
print()
print(ventas.dtypes)
print()
print(ventas.isna().sum())

(150, 6)

fecha                  str
region                 str
producto               str
unidades               str
precio_unitario    float64
cliente_id             str
dtype: object

fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64


**Filas que parecen inválidas**

unidades sale como texto (str) en vez de número porque hay una fila con "na" escrito, y hay 2 unidades vacías y 1 precio vacío.

Mirando los datos las filas raras son:

Unidades vacías: C004 (2026-02-01) y C018 (2026-02-11).

Unidades con "na": C022 (2026-02-15).

Unidades a 0 o negativas: C010 con 0 (2026-02-05), C012 con -2 (2026-02-07) y C030 con 0 (2026-02-19).

Precio vacío: C025 (2026-02-17).

Precio a 0 o negativo: C015 con 0 (2026-02-09), C030 con -1 (2026-02-19), C007 con -5 (2026-02-03) y C020 con -32 (2026-02-13).

En total son 10 filas (la de C030 tiene mal las dos cosas).

**Parte 2 - Validación**

In [4]:
def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = frame.copy()
    df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")
    df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")

    ok = (df["unidades"] > 0) & (df["precio_unitario"] > 0)

    validos = df[ok].copy()
    errores = df[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores


validos, errores = validar_ventas(ventas)
print("válidas:", len(validos), "| inválidas:", len(errores))
errores

válidas: 140 | inválidas: 10


,fecha,region,producto,unidades,precio_unitario,cliente_id
3,2026-02-05,Norte,Gateway,0.0,120.0,C010
41,2026-02-09,Oeste,Cable Kit,4.0,0.0,C015
54,2026-02-01,Sur,Sensor B,NaN,32.0,C004
63,2026-02-11,Norte,Sensor A,NaN,19.5,C018
66,2026-02-17,Oeste,Hub IoT,5.0,NaN,C025
84,2026-02-15,Este,Gateway,NaN,120.0,C022
88,2026-02-07,Este,Hub IoT,-2.0,85.0,C012
109,2026-02-19,Norte,Cable Kit,0.0,-1.0,C030
134,2026-02-03,Sur,Sensor A,3.0,-5.0,C007
149,2026-02-13,Sur,Sensor B,8.0,-32.0,C020


Me salen 140 válidas y 10 inválidas, que coincide con el checkpoint del enunciado y con las filas que había visto en el diagnóstico.

Con errors="coerce" el "na" se convierte en NaN, y no hace falta comprobar los nulos aparte porque NaN > 0 da False.

**Parte 3 - Agregaciones**

In [5]:
# importe total por region
por_region = validos.groupby("region")["importe"].sum().sort_values(ascending=False)
por_region

region
Este     33122.53
Norte    22914.63
Sur      21396.35
Oeste    19164.66
Name: importe, dtype: float64

In [6]:
# top 3 productos
top_productos = validos.groupby("producto")["importe"].sum().sort_values(ascending=False).head(3)
top_productos

producto
Gateway     40440.00
Hub IoT     33058.66
Sensor B    11568.66
Name: importe, dtype: float64

In [7]:
# clientes con mas de una compra
compras = validos["cliente_id"].value_counts()
clientes_repetidos = compras[compras > 1]
clientes_repetidos

cliente_id
C027    9
C032    8
C004    8
C035    7
C013    6
C009    6
C005    5
C022    5
C038    5
C010    4
C018    4
C002    4
C040    4
C025    4
C015    4
C024    4
C023    4
C034    4
C037    4
C016    3
C003    3
C007    3
C011    3
C029    3
C033    3
C030    3
C028    2
C036    2
C021    2
C039    2
C031    2
C001    2
C020    2
C017    2
Name: count, dtype: int64

**Parte 4 - Exportación**

In [8]:
validos.to_csv(DATA_DIR / "ventas_limpias.csv", index=False)

calidad = {
    "filas_totales": len(ventas),
    "filas_validas": len(validos),
    "filas_invalidas": len(errores),
    "importe_total": float(validos["importe"].sum()),
}

with open(DATA_DIR / "calidad_datos.json", "w", encoding="utf-8") as f:
    json.dump(calidad, f, indent=2)

calidad

{'filas_totales': 150,
 'filas_validas': 140,
 'filas_invalidas': 10,
 'importe_total': 96598.17000000001}

**Extensión - ventas por semana**

In [9]:
validos["fecha"] = pd.to_datetime(validos["fecha"])
validos["semana"] = validos["fecha"].dt.isocalendar().week
validos.groupby("semana")["importe"].sum()

semana
2      6058.74
3     12581.58
4      7762.50
5      5042.46
6     14941.88
7      5748.00
8     11914.87
9      5071.00
10     4567.92
11     6416.92
12     5513.00
13     4095.00
14     6884.30
Name: importe, dtype: float64